In [1]:
import sys
import os
import glob
import pandas as pd
import time
import subprocess

import scanpy as sc

In [2]:
output_dir = "/nfs/turbo/umms-indikar/shared/projects/HSC/geo_submission/"

# Count Matrices

In [3]:
%%time

pipeline_inputs = {
    'bone_marrow' : '/nfs/turbo/umms-indikar/shared/projects/HSC/pipeline_outputs/bm_full/anndata/bone_marrow.h5ad',
    'reprogram' : '/nfs/turbo/umms-indikar/shared/projects/HSC/pipeline_outputs/hsc_epi2me_full/anndata/gene_adata.h5ad',
    'fibroblast' : '/nfs/turbo/umms-indikar/shared/projects/HSC/pipeline_outputs/cc_fibroblast_full/anndata/cc_fibroblast.h5ad'
}

for key, file_path in pipeline_inputs.items():
    start = time.time()

    print(f"\n ======== {key}  ======== ")
    print(f"\tfile path: {file_path}")

    output_path = f"{output_dir}{key}.h5ad"
    print(f"\tsaving to: {output_path}")

    adata = sc.read_h5ad(file_path)

    # remove metadata to make raw format consistent
    del adata.obs
    del adata.var

    n_cells, n_genes = adata.shape
    print(f"\tnumber of cells: {n_cells}")
    print(f"\tnumber of genes: {n_genes}")

    adata.write(output_path)

    end = time.time()
    minutes = (end - start) / 60
    print(f"\ttotal time: {minutes:.2f} minutes")


print()    


 ======== bone_marrow  ======== 
	file path: /nfs/turbo/umms-indikar/shared/projects/HSC/pipeline_outputs/bm_full/anndata/bone_marrow.h5ad
	saving to: /nfs/turbo/umms-indikar/shared/projects/HSC/geo_submission/bone_marrow.h5ad
	number of cells: 6269
	number of genes: 26440
	total time: 0.04 minutes

 ======== reprogram  ======== 
	file path: /nfs/turbo/umms-indikar/shared/projects/HSC/pipeline_outputs/hsc_epi2me_full/anndata/gene_adata.h5ad
	saving to: /nfs/turbo/umms-indikar/shared/projects/HSC/geo_submission/reprogram.h5ad
	number of cells: 11403
	number of genes: 25311
	total time: 0.06 minutes

 ======== fibroblast  ======== 
	file path: /nfs/turbo/umms-indikar/shared/projects/HSC/pipeline_outputs/cc_fibroblast_full/anndata/cc_fibroblast.h5ad
	saving to: /nfs/turbo/umms-indikar/shared/projects/HSC/geo_submission/fibroblast.h5ad
	number of cells: 8963
	number of genes: 23635
	total time: 0.04 minutes

CPU times: user 435 ms, sys: 1.25 s, total: 1.69 s
Wall time: 8.23 s


# Metadata 

In [4]:
%%time

pipeline_inputs = {
    'bone_marrow' : '/nfs/turbo/umms-indikar/shared/projects/HSC/pipeline_outputs/bm_full/anndata/bone_marrow.h5ad',
    'reprogram' : '/nfs/turbo/umms-indikar/shared/projects/HSC/pipeline_outputs/hsc_epi2me_full/anndata/gene_adata.h5ad',
    'fibroblast' : '/nfs/turbo/umms-indikar/shared/projects/HSC/pipeline_outputs/cc_fibroblast_full/anndata/cc_fibroblast.h5ad'
}

records = []

for key, file_path in pipeline_inputs.items():
    start = time.time()

    print(f"\n ======== {key}  ======== ")
    print(f"\tfile path: {file_path}")

    adata = sc.read_h5ad(file_path)

    # Collect key and obs_names
    records.extend([(key, name) for name in adata.obs_names])

    end = time.time()
    minutes = (end - start) / 60
    print(f"\ttotal time: {minutes:.2f} minutes")

# Create the dataframe
df = pd.DataFrame(records, columns=["dataset", "barcode"])
print()
print(df['dataset'].value_counts().to_string())
print()
print(f"{df.shape=}")
print()
df.head()


 ======== bone_marrow  ======== 
	file path: /nfs/turbo/umms-indikar/shared/projects/HSC/pipeline_outputs/bm_full/anndata/bone_marrow.h5ad
	total time: 0.01 minutes

 ======== reprogram  ======== 
	file path: /nfs/turbo/umms-indikar/shared/projects/HSC/pipeline_outputs/hsc_epi2me_full/anndata/gene_adata.h5ad
	total time: 0.02 minutes

 ======== fibroblast  ======== 
	file path: /nfs/turbo/umms-indikar/shared/projects/HSC/pipeline_outputs/cc_fibroblast_full/anndata/cc_fibroblast.h5ad
	total time: 0.01 minutes

dataset
reprogram      11403
fibroblast      8963
bone_marrow     6269

df.shape=(26635, 2)

CPU times: user 278 ms, sys: 719 ms, total: 997 ms
Wall time: 2.78 s


,dataset,barcode
0,bone_marrow,AAACCAAAGGCGATCT-1
1,bone_marrow,AAACCAAAGGTGAACG-1
2,bone_marrow,AAACCATTCCCTGTCA-1
3,bone_marrow,AAACCATTCCGCGATG-1
4,bone_marrow,AAACCATTCCTGTCGC-1


In [5]:
%%time
# load processed metadata
file_path = "/nfs/turbo/umms-indikar/shared/projects/HSC/pipeline_outputs/integrated_anndata/three_libraries.h5ad"
adata = sc.read_h5ad(file_path)

source_map = {
    'hsc' : 'reprogram', 
    'fib' : 'fibroblast', 
    'bone_marrow' : 'bone_marrow', 
}

keep_columns = [
    'dataset', 
    'group',
    'label',
    'S_score',
    'G2M_score',
    'phase', 
    'subgroup',
]

print(adata)
print()

# gather annotated obs
obs = adata.obs.copy()

obs['dataset'] = obs['source'].map(source_map)
obs = obs[keep_columns]

# add the emebdding coords
obs['UMAP1'] = adata.obsm['X_umap'][:, 0]
obs['UMAP2'] = adata.obsm['X_umap'][:, 1]

# reformat barcode
obs = obs.reset_index(drop=False)
obs['barcode'] = obs['index'].str.replace(r'^((?:[^-]*-){2})[^-]*$', r'\1', regex=True).str.rstrip('-')
obs = obs.drop(columns='index')

print(f"{obs.shape=}")
print()
print(obs.head())

AnnData object with n_obs × n_vars = 21062 × 25739
    obs: 'dataset', 'source', 'cluster', 'group', 'label', 'S_score', 'G2M_score', 'phase', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mito', 'log1p_total_counts_mito', 'pct_counts_mito', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'filter_pass', 'bm_clusters', 'subgroup'
    var: 'gene_ids', 'feature_types', 'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'n_counts', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection', 'mito', 'filter_pass'
    uns: 'deg', 'rank_genes_groups', 'scvi_expression', 'subgroup_colors'
    obsm: 'X_paga', 'X_pca', 'X_pca_harmony', 'X_scvi

In [6]:
# merge the metadata and full barcode list
df = pd.merge(
    df, obs, 
    how='left',
    left_on=['barcode', 'dataset'],
    right_on=['barcode', 'dataset'],
)

df['qc_filter_pass'] = df['subgroup'].notna()

print(f"{df.shape=}")
outpath = f"{output_dir}metadata.csv"
df.to_csv(outpath, index=False)
df.head()

df.shape=(26635, 11)


,dataset,barcode,group,label,S_score,G2M_score,phase,subgroup,UMAP1,UMAP2,qc_filter_pass
0,bone_marrow,AAACCAAAGGCGATCT-1,bm_other,MPP,0.571801,-0.713437,S,progenitors,-0.128261,-3.079190,True
1,bone_marrow,AAACCAAAGGTGAACG-1,bm_other,MEG/ERY,-1.011905,-0.898002,G1,progenitors,0.047534,-5.844589,True
2,bone_marrow,AAACCATTCCCTGTCA-1,bm_other,CMP,-0.476190,17.373327,G2M,progenitors,-3.723126,-4.444674,True
3,bone_marrow,AAACCATTCCGCGATG-1,target,HSC,-0.671131,-0.575425,G1,HSC,-4.596651,-5.937883,True
4,bone_marrow,AAACCATTCCTGTCGC-1,bm_other,MPP,-1.030134,-0.586663,G1,progenitors,-3.788254,-4.361890,True


In [7]:
break

SyntaxError: 'break' outside loop (668683560.py, line 1)

# FASTQ

In [ ]:
def merge_fastq_gz_files(file_list, output_path):
    """
    Merges a list of .fastq.gz files using `cat` and writes to output_path.

    Parameters:
        file_list (list): List of input .fastq.gz file paths.
        output_path (str): Path to output merged file.
    """
    if not file_list:
        raise ValueError("file_list is empty.")

    if os.path.exists(output_path):
        os.remove(output_path)

    cmd = ["cat"] + file_list
    with open(output_path, 'wb') as outfile:
        subprocess.run(cmd, stdout=outfile, check=True)


def get_fastq_files(fastq_path):
    """
    Given a list of fastq.gz files or directories, return a list of absolute fastq.gz file paths.
    """
    resolved_files = []

    for path in fastq_path:
        path = os.path.abspath(path)
        if os.path.isfile(path) and path.endswith(".fastq.gz"):
            resolved_files.append(path)
        elif os.path.isdir(path):
            # Recursively find all .fastq.gz files in the directory
            files = glob.glob(os.path.join(path, "**", "*.fastq.gz"), recursive=True)
            resolved_files.extend(map(os.path.abspath, files))
        else:
            raise ValueError(f"Invalid path: {path}")
    
    return sorted(resolved_files)

In [ ]:
%%time

pipeline_inputs = {
    'bone_marrow' : '/home/cstansbu/git_repositories/hematokytos/pipelines/bm_pipeline/',
    'reprogram' : '/home/cstansbu/git_repositories/hematokytos/pipelines/hsc_pipeline/',
    'fibroblast' : '/home/cstansbu/git_repositories/hematokytos/pipelines/cc_pipeline/'
}

for key, dir_path in pipeline_inputs.items():
    start = time.time()
    
    print(f"\n ======== {key}  ======== ")
    manifest_path = f"{dir_path}fastq_paths.txt"
    
    print(f"\tinput path: {manifest_path}")
    output_path = f"{output_dir}{key}.fastq.gz"
    print(f"\tsaving to: {output_path}")
    
    df = pd.read_csv(manifest_path)
    print(f"\tnumber of locations: {len(df)}")
    
    fastq_path = df['file_path'].to_list()
    file_list = get_fastq_files(fastq_path)
    print(f"\tnumber of files: {len(file_list)}")

    print(f"\tstarting file write...")
    merge_fastq_gz_files(file_list, output_path)

    end = time.time()
    minutes = (end - start) / 60
    print(f"\ttotal time: {minutes:.2f} minutes")
    
print()    

In [ ]:
break